# 04 — Preprocessing summary

**Purpose:** summarize the outputs of Notebooks 01–03: cleaning operations, recorded issues, factor categorization, cohort filtering, and final sample composition.

**Inputs:** the six CSV files previously generated in `outputs/`:
- `dataset_clean.csv`, `data_errors.csv`, `error_examples.csv`
- `dataset_categorized.csv`
- `dataset_final.csv`, `dataset_excluded.csv`

**Output:** `outputs/preprocessing_summary.md`, a consolidated report with the same tables shown in this notebook.

This notebook does **not** clean, filter, categorize, or modify any dataset.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path("../../outputs")
files = {
    "clean": "dataset_clean.csv",
    "errors": "data_errors.csv",
    "examples": "error_examples.csv",
    "categorized": "dataset_categorized.csv",
    "final": "dataset_final.csv",
    "excluded": "dataset_excluded.csv",
}

missing_files = [name for name in files.values() if not (OUTPUT_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing preprocessing outputs: {missing_files}")

data = {name: pd.read_csv(OUTPUT_DIR / filename) for name, filename in files.items()}
clean = data["clean"]
errors = data["errors"]
examples = data["examples"]
categorized = data["categorized"]
final = data["final"]
excluded = data["excluded"]

print("All six preprocessing outputs loaded successfully.")

All six preprocessing outputs loaded successfully.


## 1. Dataset flow and consistency

In [2]:
# Check that every input participant is either retained or excluded, exactly once.
for name, frame in [("clean", clean), ("categorized", categorized),
                    ("final", final), ("excluded", excluded)]:
    if "source_excel_row" not in frame or not frame["source_excel_row"].is_unique:
        raise ValueError(f"Missing or duplicated source_excel_row in {name}")

clean_ids = set(clean["source_excel_row"])
categorized_ids = set(categorized["source_excel_row"])
final_ids = set(final["source_excel_row"])
excluded_ids = set(excluded["source_excel_row"])
assert clean_ids == categorized_ids, "Rows changed between Notebooks 01 and 02"
assert final_ids.isdisjoint(excluded_ids), "A participant is both retained and excluded"
assert final_ids | excluded_ids == categorized_ids, "Stage 03 lost or added participants"
assert final.columns.tolist() == categorized.columns.tolist(), "Final dataset columns differ from categorized input"

flow = pd.DataFrame({
    "stage": ["01 — Cleaned", "02 — Categorized", "03 — Retained", "03 — Excluded"],
    "participants": [len(clean), len(categorized), len(final), len(excluded)],
    "columns": [clean.shape[1], categorized.shape[1], final.shape[1], excluded.shape[1]],
})
flow["percent_of_categorized"] = (100 * flow["participants"] / len(categorized)).round(1)

print("Dataset flow (excluded CSV includes two extra reason columns):")
display(flow)
print(f"Retention: {len(final)}/{len(categorized)} ({len(final)/len(categorized):.1%})")
print(f"Exclusion: {len(excluded)}/{len(categorized)} ({len(excluded)/len(categorized):.1%})")

Dataset flow (excluded CSV includes two extra reason columns):


,stage,participants,columns,percent_of_categorized
0,01 — Cleaned,530,19,100.0
1,02 — Categorized,530,25,100.0
2,03 — Retained,476,25,89.8
3,03 — Excluded,54,27,10.2


Retention: 476/530 (89.8%)
Exclusion: 54/530 (10.2%)


## 2. Cleaning and validation summary

In [3]:
# An issue is one log entry, not necessarily one affected participant.
issue_summary = (errors.groupby(["issue", "action"], dropna=False)
                 .size().reset_index(name="entries")
                 .sort_values("entries", ascending=False).reset_index(drop=True))

issue_by_column = (errors.groupby("column", dropna=False).size()
                   .reset_index(name="entries")
                   .sort_values("entries", ascending=False).reset_index(drop=True))

logged_rows = set(errors["source_excel_row"].dropna().astype(int))
example_rows = set(examples["source_excel_row"].astype(int))
if logged_rows != example_rows:
    raise ValueError("error_examples.csv does not match the rows recorded in data_errors.csv")

print(f"Log entries: {len(errors)}")
print(f"Participants with at least one logged issue: {len(logged_rows)}")
print(f"Rows in error_examples.csv: {len(examples)}")
print("\nEntries by issue type and action:")
display(issue_summary)
print("\nEntries by column:")
display(issue_by_column)
print("\nNote: issue counts can overlap within the same participant.")

Log entries: 183
Participants with at least one logged issue: 155
Rows in error_examples.csv: 155

Entries by issue type and action:


,issue,action,entries
0,text_number,converted_to_numeric,97
1,missing_value,kept_missing,20
2,identical_hrv_profile,review_no_change,16
3,outside_review_range,review_no_change,14
4,hr_rr_factor_10_pattern,review_no_change,11
5,bmi_disagreement,recomputed_from_weight_and_height,9
6,height_disagreement,corrected_from_height_cm,9
7,hr_rr_disagreement,review_no_change,4
8,empty_column,removed_column,2
9,empty_metadata_column,retained_for_traceability,1



Entries by column:


,column,entries
0,t2m_pre_rmssd,105
1,hrv_profile,16
2,t2m_pre_mean_hr/t2m_pre_mean_rr,15
3,t2m_pre_sdnn,14
4,t2m_pre_mean_rr,10
5,height_m,9
6,imc,9
7,t2m_pre_mean_hr,2
8,Unnamed: 0,1
9,Unnamed: 1,1



Note: issue counts can overlap within the same participant.


## 3. Factor categorization before and after filtering

In [4]:
factors = ["age_group", "sex_category", "sbp_category", "dbp_category", "bmi_category", "bmi_reference"]
missing_factors = [factor for factor in factors if factor not in categorized or factor not in final]
if missing_factors:
    raise ValueError(f"Missing category columns: {missing_factors}")

factor_tables = {}
for factor in factors:
    before = categorized[factor].fillna("Missing").value_counts(dropna=False)
    after = final[factor].fillna("Missing").value_counts(dropna=False)
    table = pd.DataFrame({"categorized_n": before, "retained_n": after}).fillna(0).astype(int)
    table["categorized_pct"] = (100 * table["categorized_n"] / len(categorized)).round(1)
    table["retained_pct"] = (100 * table["retained_n"] / len(final)).round(1)
    table.index.name = factor
    factor_tables[factor] = table.reset_index()
    print(f"\n{factor}")
    display(factor_tables[factor])


age_group


,age_group,categorized_n,retained_n,categorized_pct,retained_pct
0,60-69,219,198,41.3,41.6
1,70-79,261,241,49.2,50.6
2,80+,40,37,7.5,7.8
3,<60,10,0,1.9,0.0



sex_category


,sex_category,categorized_n,retained_n,categorized_pct,retained_pct
0,Female,366,333,69.1,70.0
1,Male,164,143,30.9,30.0



sbp_category


,sbp_category,categorized_n,retained_n,categorized_pct,retained_pct
0,<140 mmHg,395,351,74.5,73.7
1,>=140 mmHg,135,125,25.5,26.3


... (2 outputs left out)


bmi_category


,bmi_category,categorized_n,retained_n,categorized_pct,retained_pct
0,Obesity,186,174,35.1,36.6
1,Overweight,166,145,31.3,30.5
2,Normal,155,136,29.2,28.6
3,Underweight,23,21,4.3,4.4



bmi_reference


,bmi_reference,categorized_n,retained_n,categorized_pct,retained_pct
0,MINSAL adult (<60),10,0,1.9,0.0
1,MINSAL older person (>=60),520,476,98.1,100.0


## 4. Exclusion criteria and multiple reasons

In [5]:
needed = ["exclusion_count", "exclusion_reasons"]
if any(name not in excluded for name in needed):
    raise ValueError("Exclusion count/reasons columns are missing")

# Split the exact separator written by Notebook 03.
all_reasons = excluded["exclusion_reasons"].fillna("").str.split(" | ", regex=False)
if not (all_reasons.str.len() == excluded["exclusion_count"]).all():
    raise ValueError("Some exclusion_count values do not match the listed reasons")

reason_summary = (all_reasons.explode().value_counts()
                  .rename_axis("reason").reset_index(name="participants"))
reason_summary["percent_of_excluded"] = (
    100 * reason_summary["participants"] / len(excluded)
).round(1)

multiplicity = (excluded["exclusion_count"].value_counts().sort_index()
                .rename_axis("reasons_per_participant")
                .reset_index(name="participants"))

reason_combinations = (excluded["exclusion_reasons"].value_counts()
                       .rename_axis("combined_reasons").reset_index(name="participants"))

print("Exclusion reasons (not mutually exclusive):")
display(reason_summary)
print("\nNumber of distinct reasons per excluded participant:")
display(multiplicity)
print("\nObserved combinations of exclusion reasons:")
display(reason_combinations)
print(f"\nExcluded participants with multiple reasons: {(excluded['exclusion_count'] > 1).sum()}")

Exclusion reasons (not mutually exclusive):


,reason,participants,percent_of_excluded
0,IDENTICAL_HRV_PROFILE,16,29.6
1,HR_RR_FACTOR_10,11,20.4
2,MISSING_T2M_PRE_SDNN,11,20.4
3,AGE_BELOW_60,10,18.5
4,MISSING_T2M_PRE_RMSSD,8,14.8
5,HR_RR_MAJOR_MISMATCH,2,3.7
6,MISSING_T2M_PRE_MEAN_HR,1,1.9



Number of distinct reasons per excluded participant:


,reasons_per_participant,participants
0,1,49
1,2,5



Observed combinations of exclusion reasons:


,combined_reasons,participants
0,IDENTICAL_HRV_PROFILE,15
1,HR_RR_FACTOR_10,11
2,AGE_BELOW_60,9
3,MISSING_T2M_PRE_SDNN,7
4,MISSING_T2M_PRE_SDNN | MISSING_T2M_PRE_RMSSD,4
5,MISSING_T2M_PRE_RMSSD,4
6,HR_RR_MAJOR_MISMATCH,2
7,AGE_BELOW_60 | IDENTICAL_HRV_PROFILE,1
8,MISSING_T2M_PRE_MEAN_HR,1



Excluded participants with multiple reasons: 5


## 5. Final sample composition and factorial group sizes

In [6]:
age_by_sex = pd.crosstab(final["age_group"], final["sex_category"])
sbp_by_bmi = pd.crosstab(final["sbp_category"], final["bmi_category"])
dbp_by_bmi = pd.crosstab(final["dbp_category"], final["bmi_category"])

# Each pressure measurement defines a separate four-factor analysis.
# The two pressure categories must not be merged into a new factor.
four_factor_counts = {}
for pressure_factor in ["sbp_category", "dbp_category"]:
    four_factor_counts[pressure_factor] = (
        final.groupby(
            ["age_group", "sex_category", pressure_factor, "bmi_category"],
            dropna=False
        ).size().reset_index(name="participants")
    )

print("Age group × sex (retained sample):")
display(age_by_sex)
print("\nSystolic BP category × BMI category (retained sample):")
display(sbp_by_bmi)
print("\nDiastolic BP category × BMI category (retained sample):")
display(dbp_by_bmi)

for pressure_factor, counts in four_factor_counts.items():
    print(f"\nAge × sex × {pressure_factor} × BMI")
    print("Observed combinations:", len(counts))
    print("Combinations with fewer than 5 participants:", (counts["participants"] < 5).sum())
    print("Combinations with fewer than 10 participants:", (counts["participants"] < 10).sum())
    print("Smallest observed four-factor groups:")
    display(counts.sort_values("participants").head(15))

hrv_columns = ["t2m_pre_mean_rr", "t2m_pre_mean_hr", "t2m_pre_sdnn",
               "t2m_pre_rmssd", "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf"]
missing_hrv = final[hrv_columns].isna().sum().rename("missing_n").reset_index()
missing_hrv.columns = ["hrv_variable", "missing_n"]
print("\nRemaining missing HRV values:")
display(missing_hrv)

Age group × sex (retained sample):


sex_category,Female,Male
age_group,,
60-69,150,48
70-79,156,85
80+,27,10



Systolic BP category × BMI category (retained sample):


bmi_category,Normal,Obesity,Overweight,Underweight
sbp_category,,,,
<140 mmHg,104,120,109,18
>=140 mmHg,32,54,36,3



Diastolic BP category × BMI category (retained sample):


bmi_category,Normal,Obesity,Overweight,Underweight
dbp_category,,,,
<90 mmHg,124,145,129,19
>=90 mmHg,12,29,16,2


... (2 outputs left out)


Age × sex × dbp_category × BMI
Observed combinations: 37
Combinations with fewer than 5 participants: 15
Combinations with fewer than 10 participants: 23
Smallest observed four-factor groups:


,age_group,sex_category,dbp_category,bmi_category,participants
11,60-69,Male,>=90 mmHg,Normal,1
29,70-79,Male,>=90 mmHg,Underweight,1
21,70-79,Female,>=90 mmHg,Underweight,1
18,70-79,Female,>=90 mmHg,Normal,1
25,70-79,Male,<90 mmHg,Underweight,1
33,80+,Female,<90 mmHg,Underweight,1
28,70-79,Male,>=90 mmHg,Overweight,2
10,60-69,Male,<90 mmHg,Underweight,2
35,80+,Male,<90 mmHg,Obesity,2
13,60-69,Male,>=90 mmHg,Overweight,3



Remaining missing HRV values:


,hrv_variable,missing_n
0,t2m_pre_mean_rr,0
1,t2m_pre_mean_hr,0
2,t2m_pre_sdnn,0
3,t2m_pre_rmssd,0
4,t2m_pre_hf,0
5,t2m_pre_lf,0
6,t2m_pre_vlf,0


## 6. Export consolidated summary

In [7]:
# A single readable report; no dataset is rewritten.
report = [
    "# HRV preprocessing summary",
    "",
    "Based on the existing outputs of Notebooks 01, 02 and 03.",
    "",
    "## Dataset flow",
    flow.to_markdown(index=False),
    "",
    f"Retention: {len(final)}/{len(categorized)} ({len(final)/len(categorized):.1%})",
    f"Exclusion: {len(excluded)}/{len(categorized)} ({len(excluded)/len(categorized):.1%})",
    "",
    "## Cleaning and validation",
    f"Log entries: {len(errors)}. Unique participants with logged issues: {len(logged_rows)}.",
    f"Original error examples: {len(examples)} rows.",
    "",
    issue_summary.to_markdown(index=False),
    "",
    "### Issues by column",
    issue_by_column.to_markdown(index=False),
    "",
    "## Factor categories",
]
for factor, table in factor_tables.items():
    report.extend(["", f"### {factor}", table.to_markdown(index=False)])

report.extend([
    "", "## Exclusion reasons",
    "Reasons are not mutually exclusive; counts should not be summed to obtain excluded participants.",
    reason_summary.to_markdown(index=False),
    "", "### Number of reasons per participant",
    multiplicity.to_markdown(index=False),
    "", "### Observed exclusion combinations",
    reason_combinations.assign(
        combined_reasons=lambda frame: frame["combined_reasons"].str.replace("|", r"\|", regex=False)
    ).to_markdown(index=False),
    "", "## Final sample: age by sex",
    age_by_sex.to_markdown(),
    "", "## Final sample: systolic BP by BMI",
    sbp_by_bmi.to_markdown(),
    "", "## Final sample: diastolic BP by BMI",
    dbp_by_bmi.to_markdown(),
    "", "## Four-factor combinations",
    "Age × sex × pressure × BMI is summarized separately for systolic and diastolic pressure.",
])

for pressure_factor, counts in four_factor_counts.items():
    report.extend([
        "",
        f"### Age × sex × {pressure_factor} × BMI",
        f"Observed combinations: {len(counts)}; "
        f"fewer than five participants: {(counts['participants'] < 5).sum()}; "
        f"fewer than ten: {(counts['participants'] < 10).sum()}.",
        "",
        "Smallest observed groups:",
        counts.sort_values("participants").head(15).to_markdown(index=False),
    ])

report.extend([
    "", "## Missing HRV values in the final sample",
    missing_hrv.to_markdown(index=False),
    "",
])

SUMMARY_FILE = OUTPUT_DIR / "preprocessing_summary.md"
SUMMARY_FILE.write_text("\n".join(report), encoding="utf-8")
print(f"Saved: {SUMMARY_FILE}")
print("All input datasets remain unchanged.")

Saved: ../../outputs/preprocessing_summary.md
All input datasets remain unchanged.
